# SIMILIS baseline: data loading sanity-check

Показываем воспроизводимый способ получения данных, пути к `csv` и изображениям, а также проверяем, что `raw`-часть читается корректно.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_URL = 'https://drive.usercontent.google.com/download?id=11yX9-6qRPLk53qiy1I9fNg9Xfd_kYvgd&export=download&authuser=0'
ZIP_PATH = RAW_DIR / 'cu_data.zip'
EXTRACT_DIR = RAW_DIR / 'cu_data'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('DATA_URL:', DATA_URL)
print('ZIP_PATH exists:', ZIP_PATH.exists())
print('EXTRACT_DIR exists:', EXTRACT_DIR.exists())

PROJECT_ROOT: /Users/artemvardanan/Downloads/DL_project1
RAW_DIR: /Users/artemvardanan/Downloads/DL_project1/data/raw
DATA_URL: https://drive.usercontent.google.com/download?id=11yX9-6qRPLk53qiy1I9fNg9Xfd_kYvgd&export=download&authuser=0
ZIP_PATH exists: True
EXTRACT_DIR exists: True


## Reproducible data access

Воспроизводимый способ получения данных в проекте:

1. скачать архив `cu_data.zip` по ссылке `DATA_URL` в `data/raw/`;
2. распаковать его в `data/raw/cu_data/`;
3. ниже указать путь к корню изображений и путь к основному `csv`.

In [3]:
download_cmd = f'curl -L "{DATA_URL}" -o "{ZIP_PATH}"'
print(download_cmd)

curl -L "https://drive.usercontent.google.com/download?id=11yX9-6qRPLk53qiy1I9fNg9Xfd_kYvgd&export=download&authuser=0" -o "/Users/artemvardanan/Downloads/DL_project1/data/raw/cu_data.zip"


In [4]:
raw_csv_candidates = sorted(list(EXTRACT_DIR.rglob('*.csv'))) if EXTRACT_DIR.exists() else []

# Ignore macOS service files from extracted archives.
csv_candidates = [
    p for p in raw_csv_candidates
    if '__MACOSX' not in p.parts and not p.name.startswith('._')
]

image_candidates = []
if EXTRACT_DIR.exists():
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff', '*.webp'):
        image_candidates.extend(list(EXTRACT_DIR.rglob(ext)))
    image_candidates = sorted(set(image_candidates))

print('CSV candidates found (raw):', len(raw_csv_candidates))
print('CSV candidates found (filtered):', len(csv_candidates))
for path in csv_candidates[:10]:
    print('-', path)

print('Image files found:', len(image_candidates))


CSV candidates found (raw): 2
CSV candidates found (filtered): 1
- /Users/artemvardanan/Downloads/DL_project1/data/raw/cu_data/cu_data/selected_by_name_iimk_subset_public.csv
Image files found: 2776


In [5]:
if not csv_candidates:
    raise FileNotFoundError('No valid CSV files found under extracted dataset directory')

# Prefer expected public subset file if present, otherwise fallback to first valid CSV.
preferred_name = 'selected_by_name_iimk_subset_public.csv'
CSV_PATH = next((p for p in csv_candidates if p.name == preferred_name), csv_candidates[0])

df = pd.read_csv(CSV_PATH)

print('Main CSV path:', CSV_PATH)
print('CSV rows:', len(df))
display(df.head())

Main CSV path: /Users/artemvardanan/Downloads/DL_project1/data/raw/cu_data/cu_data/selected_by_name_iimk_subset_public.csv
CSV rows: 1389


,Unnamed: 0,code,name,description,material,size,fragm,cultlayer,execorg,survyear
0,40,М102-2012-1-0494,Изразец,Изразца красноглиняного с белой поливой и коба...,Керамика,"7+ х 3,5+ х 1,1+ см",Фрагмент,Пестрая мешаная супесь,ИИМК РАН,2012.0
1,41,М102-2012-1-0496,Изразец,Изразца красноглиняного плоского с белой полив...,Керамика,"9,9+ х 6,4+ х 1,3+ см",Фрагмент,Пестрая мешаная супесь,ИИМК РАН,2012.0
2,49,М102-2012-1-0529,Изразец,Изразца-перемычки красноглиняного с белой поли...,Керамика,"4,5+ х 4,3 х 4,2+ см",Фрагмент,Пестрая мешаная супесь,ИИМК РАН,2012.0
3,59,М102-2012-1-0585,Тарелка,Сосуда (тарелки ?) фаянсового с белой поливой ...,Фаянс,D? см,Фрагмент,Дерн со щепой,ИИМК РАН,2012.0
4,63,М102-2012-1-0603,Изразец,Изразца (перемычки?) красноглиняного с белой п...,Керамика,"3,8+ х 2,8 х 5,1+ см",Фрагмент,Дерн со щепой,ИИМК РАН,2012.0


In [6]:
image_root = EXTRACT_DIR
print('Image root path:', image_root)
print('Total image files on disk:', len(image_candidates))

Image root path: /Users/artemvardanan/Downloads/DL_project1/data/raw/cu_data
Total image files on disk: 2776
